# ADD 2026 Track 2 — MOSS-Audio 4B LoRA

이 노트북은 `r=16`과 `r=32`를 각각 1 epoch 학습·평가하고, `final_score_100`이 더 높은 rank만 베이스 모델에서 3 epochs로 다시 학습합니다. 별도 가상환경이나 패키지 설치/버전 고정은 하지 않습니다. A100 런타임을 권장합니다.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/youhan200203/ADD2026-track2-baseline.git"
PROJECT_DIR = Path("/content/ADD2026-track2-baseline")
MOSS_REPO = Path("/content/MOSS-Audio")
MODEL_ID = "OpenMOSS-Team/MOSS-Audio-4B-Instruct"
MODEL_DIR = Path("/content/MOSS-Audio-4B-Instruct")
DATASET_FILE_ID = "1QQIEf5lE5I5dIpKrnxGW8noj71zxZmd8"
DATA_ARCHIVE = Path("/content/track2_training_datasets.zip")
DATA_ROOT = Path("/content/track2_training_datasets")
MOSS_DATA_DIR = PROJECT_DIR / "data/moss"

DRIVE_ROOT = Path("/content/drive/MyDrive/ADD2026/moss_audio_4b")
R16_DIR = DRIVE_ROOT / "r16_epoch1"
R32_DIR = DRIVE_ROOT / "r32_epoch1"
R16_PRED = DRIVE_ROOT / "r16_epoch1_dev.jsonl"
R32_PRED = DRIVE_ROOT / "r32_epoch1_dev.jsonl"
R16_EVAL = DRIVE_ROOT / "r16_epoch1_eval.json"
R32_EVAL = DRIVE_ROOT / "r32_epoch1_eval.json"

MAX_LEN = 4096
MAX_NEW_TOKENS = 2048
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
SEED = 42
print({"model": MODEL_ID, "ranks": [16, 32], "max_len": MAX_LEN})

In [ ]:
import importlib.util
import shutil
import sys

required_modules = [
    "accelerate", "bert_score", "gdown", "huggingface_hub",
    "librosa", "peft", "soundfile", "torch", "transformers",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
missing_commands = [name for name in ("git", "accelerate") if shutil.which(name) is None]
if missing_modules or missing_commands:
    raise RuntimeError({"missing_modules": missing_modules, "missing_commands": missing_commands})

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required")
print({"python": sys.version.split()[0], "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## 코드와 공식 MOSS-Audio 저장소

이미 clone되어 있으면 자동 pull하지 않고 현재 파일을 그대로 재사용합니다.

In [ ]:
if PROJECT_DIR.is_dir():
    print(f"[reuse] {PROJECT_DIR}")
else:
    !git clone {PROJECT_REPO} {PROJECT_DIR}

if MOSS_REPO.is_dir():
    print(f"[reuse] {MOSS_REPO}")
else:
    !git clone https://github.com/OpenMOSS/MOSS-Audio.git {MOSS_REPO}

if not (PROJECT_DIR / "prepare_moss_data.py").is_file():
    raise FileNotFoundError(PROJECT_DIR / "prepare_moss_data.py")
if not (MOSS_REPO / "finetune/finetune.py").is_file():
    raise FileNotFoundError(MOSS_REPO / "finetune/finetune.py")

## 데이터 다운로드와 연결

압축 내부의 실제 루트인 `track2_training_datasets/{train,dev,labels}`를 사용합니다. 프로젝트에는 세 디렉터리의 symlink만 만듭니다.

In [ ]:
import gdown
import os
import zipfile

if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    if not DATA_ARCHIVE.is_file():
        gdown.download(
            id=DATASET_FILE_ID,
            output=DATA_ARCHIVE.as_posix(),
            quiet=False,
        )
    if not zipfile.is_zipfile(DATA_ARCHIVE):
        raise RuntimeError(f"Downloaded dataset is not a ZIP file: {DATA_ARCHIVE}")
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall("/content")

if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    raise RuntimeError(f"Expected track2_training_datasets/train, dev, labels under {DATA_ROOT}")

project_data = PROJECT_DIR / "data"
project_data.mkdir(exist_ok=True)
for name in ("train", "dev", "labels"):
    target = project_data / name
    source = DATA_ROOT / name
    if target.is_symlink():
        if target.resolve() != source.resolve():
            raise RuntimeError(f"Unexpected symlink target: {target} -> {target.resolve()}")
    elif target.exists():
        raise RuntimeError(f"Refusing to replace existing path: {target}")
    else:
        os.symlink(source, target, target_is_directory=True)

print({name: len(list((DATA_ROOT / name).glob("*.wav"))) for name in ("train", "dev")})

## MOSS-Audio 4B 모델 다운로드

Hugging Face의 다운로드 진행률과 ETA가 표시됩니다.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_DIR,
)
print(f"[ready] {MODEL_DIR}")

## 학습·평가 manifest 생성

`prepare_moss_data.py`는 모든 train/dev 행, WAV 존재 여부, label, XML Decision/Evidence를 검사하며 중복 제거는 하지 않습니다. 기존 평가기는 `data/swift/dev.jsonl`을 정답 subset으로 사용합니다.

In [ ]:
import os
os.chdir(PROJECT_DIR)
print(f"[cwd] {Path.cwd()}")
!python -u prepare_moss_data.py --data-root data --output-dir {MOSS_DATA_DIR}
!python -u prepare_data.py

## 1차 실험 A — rank 16, 1 epoch

effective batch size는 `1 × 8 = 8`, LoRA alpha는 32입니다. 완료된 adapter가 Drive에 있으면 재학습하지 않습니다.

In [ ]:
if (R16_DIR / "adapter_config.json").is_file():
    print(f"[skip] completed adapter: {R16_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 accelerate launch {MOSS_REPO}/finetune/finetune.py \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {R16_DIR} --use_lora --lora_rank 16 --lora_alpha 32 \
      --attn_implementation sdpa --bf16 True --tf32 True \
      --gradient_checkpointing True --per_device_train_batch_size 1 \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} \
      --learning_rate {LEARNING_RATE} --num_train_epochs 1 \
      --lr_scheduler_type cosine --warmup_ratio 0.03 --logging_steps 10 \
      --save_strategy epoch --save_total_limit 1 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 \
      --max_len {MAX_LEN} --seed {SEED}

In [ ]:
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} \
  --adapter-dir {R16_DIR} --audio-dir data/dev --output {R16_PRED} \
  --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {R16_PRED} --subset data/swift/dev.jsonl \
  --out-json {R16_EVAL} --bert-batch-size 32 --bert-device cuda:0

## 1차 실험 B — rank 32, 1 epoch

나머지 조건은 rank 16 실험과 같고 LoRA alpha만 64입니다.

In [ ]:
if (R32_DIR / "adapter_config.json").is_file():
    print(f"[skip] completed adapter: {R32_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 accelerate launch {MOSS_REPO}/finetune/finetune.py \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {R32_DIR} --use_lora --lora_rank 32 --lora_alpha 64 \
      --attn_implementation sdpa --bf16 True --tf32 True \
      --gradient_checkpointing True --per_device_train_batch_size 1 \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} \
      --learning_rate {LEARNING_RATE} --num_train_epochs 1 \
      --lr_scheduler_type cosine --warmup_ratio 0.03 --logging_steps 10 \
      --save_strategy epoch --save_total_limit 1 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 \
      --max_len {MAX_LEN} --seed {SEED}

In [ ]:
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} \
  --adapter-dir {R32_DIR} --audio-dir data/dev --output {R32_PRED} \
  --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {R32_PRED} --subset data/swift/dev.jsonl \
  --out-json {R32_EVAL} --bert-batch-size 32 --bert-device cuda:0

## 1-epoch 승자 자동 선택

대회 공식 결합 지표인 `final_score_100`만 선택 기준으로 사용합니다. 동점이면 파라미터가 적은 rank 16을 선택합니다. 표의 `mean_bertscore_f1`은 탐지 정오답과 무관한 전체 Evidence BERTScore-F1 평균입니다.

In [ ]:
import json

def compact_result(rank, path):
    result = json.loads(path.read_text(encoding="utf-8"))
    real = result["real"]
    fake = result["fake"]
    total = real["total"] + fake["total"]
    mean_bert = (
        real["mean_bertscore_f1_all"] * real["total"]
        + fake["mean_bertscore_f1_all"] * fake["total"]
    ) / total
    return {
        "rank": rank,
        "balanced_accuracy": result["balanced_accuracy"],
        "mean_bertscore_f1": mean_bert,
        "final_score_100": result["final_score_100"],
    }

comparison = [compact_result(16, R16_EVAL), compact_result(32, R32_EVAL)]
comparison.sort(key=lambda row: (-row["final_score_100"], row["rank"]))
WINNER_RANK = comparison[0]["rank"]
WINNER_ALPHA = 2 * WINNER_RANK
WINNER_DIR = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3"
WINNER_PRED = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3_dev.jsonl"
WINNER_EVAL = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3_eval.json"
print(json.dumps({"experiments": comparison, "winner_rank": WINNER_RANK}, indent=2))

## 본 실험 — 승자 rank, 3 epochs

1-epoch adapter를 이어 붙이지 않고 동일한 베이스 4B에서 선택된 rank로 정확히 3 epochs를 새로 학습합니다. 각 epoch checkpoint는 마지막 하나만 유지되고 최종 adapter는 Drive에 저장됩니다.

In [ ]:
if (WINNER_DIR / "adapter_config.json").is_file():
    print(f"[skip] completed adapter: {WINNER_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 accelerate launch {MOSS_REPO}/finetune/finetune.py \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {WINNER_DIR} --use_lora --lora_rank {WINNER_RANK} \
      --lora_alpha {WINNER_ALPHA} --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True \
      --per_device_train_batch_size 1 \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} \
      --learning_rate {LEARNING_RATE} --num_train_epochs 3 \
      --lr_scheduler_type cosine --warmup_ratio 0.03 --logging_steps 10 \
      --save_strategy epoch --save_total_limit 1 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 \
      --max_len {MAX_LEN} --seed {SEED}

In [ ]:
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} \
  --adapter-dir {WINNER_DIR} --audio-dir data/dev --output {WINNER_PRED} \
  --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {WINNER_PRED} --subset data/swift/dev.jsonl \
  --out-json {WINNER_EVAL} --bert-batch-size 32 --bert-device cuda:0

In [ ]:
final_result = compact_result(WINNER_RANK, WINNER_EVAL)
summary = {
    "model": MODEL_ID,
    "one_epoch_experiments": comparison,
    "winner_rank": WINNER_RANK,
    "winner_three_epochs": final_result,
    "adapter_dir": WINNER_DIR.as_posix(),
    "predictions": WINNER_PRED.as_posix(),
}
summary_path = DRIVE_ROOT / "experiment_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"[saved] {summary_path}")